## 🎯 Learning Objectives
* Understand the core components and high-level architecture of an agentic e-commerce RAG system.
* Identify key use cases for agentic AI in e-commerce: product Q&A, recommendations, and order queries.
* Grasp the role of multi-agent collaboration, specifically using AutoGen, for handling complex e-commerce tasks.
* Set up a basic development environment for AutoGen and confirm its functionality.


# Welcome to PRJ-02: Agentic RAG with AutoGen for eCommerce!

## Lesson 01: System Overview: Product Q&A, Recommendations, Order Queries

Welcome, capstone developers, to the first lesson of PRJ-02! This project is the culmination of your journey through `ADV-02` (Advanced Agentic Design) and `RAG-01` (Foundational RAG), applying those cutting-edge concepts to build a sophisticated multi-agent e-commerce catalog search and query routing assistant using AutoGen.

### Why This Project Matters

In the rapidly evolving landscape of e-commerce, customer experience is paramount. Traditional search and support systems often fall short when dealing with complex, multi-faceted customer inquiries. Imagine a system that can not only answer specific product questions but also provide personalized recommendations and handle order-related queries, all while understanding context and intent. This is the power of Agentic AI.

By building an agentic RAG system, you'll empower e-commerce platforms with:

*   **Enhanced Customer Satisfaction:** Instant, accurate, and personalized responses to a wide range of inquiries.
*   **Operational Efficiency:** Automating routine customer service tasks, freeing up human agents for more complex issues.
*   **Scalability:** A system that can handle increasing query volumes without proportional increases in human resources.
*   **Competitive Advantage:** Delivering a superior, intelligent shopping experience that sets businesses apart.

### What to Expect in This Lesson

This introductory lesson, `PRJ02-L01`, will lay the groundwork by providing a high-level overview of the entire system we aim to build. We'll explore the core functionalities – product Q&A, recommendations, and order queries – and discuss how a multi-agent architecture, orchestrated by AutoGen, can seamlessly integrate these capabilities. Think of this as sketching the blueprint before we start laying the bricks.

By the end of this lesson, you'll have a clear understanding of the project's scope, its key components, and how they interact to form a powerful e-commerce assistant. Let's get started!


## Prerequisites and Tools Check

Before we dive deeper, let's ensure you have the necessary foundational knowledge and tools ready. This project builds directly upon:

*   **ADV-02: Advanced Agentic Design:** Understanding agent roles, communication protocols, and complex task decomposition.
*   **RAG-01: Foundational RAG:** Proficiency in Retrieval Augmented Generation, including vector databases, embedding models, and retrieval strategies.

### Key Tools for 2026

Our project will leverage a modern stack, focusing on robust and scalable solutions:

*   **Python 3.10+:** The primary programming language.
*   **AutoGen (v0.2.x or later stable):** The multi-agent framework for orchestration and communication.
*   **OpenAI API (or compatible LLM provider):** For powerful Large Language Models (LLMs) that drive agent reasoning. We'll primarily use OpenAI's models, but the architecture is designed to be LLM-agnostic.
*   **Qdrant (or similar Vector Database):** For efficient storage and retrieval of product catalog embeddings.
*   **Python-dotenv:** For securely managing API keys and environment variables.
*   **LangChain/LlamaIndex (optional, for RAG components):** While AutoGen can integrate RAG directly, these libraries offer advanced RAG capabilities that can be leveraged for complex retrieval strategies.

### Setting up Your Environment (Google Colab / Local)

For this project, we recommend using Google Colab for its ease of setup and access to GPU resources, especially when dealing with embedding models or larger LLMs. If you prefer a local setup, ensure you have Python 3.10+ installed and a virtual environment activated.

**Colab Specifics:**

1.  **Runtime Type:** Go to `Runtime > Change runtime type` and select `T4 GPU` for optimal performance.
2.  **API Keys:** We'll use a `.env` file for API keys. In Colab, you can upload this file or set environment variables directly.


In [ ]:
# Install necessary libraries. We're targeting stable versions as of early 2026.
# Note: Specific version numbers might be adjusted based on the latest releases.
!pip install autogen~=0.2.0 qdrant-client~=1.8.0 openai~=1.10.0 python-dotenv~=1.0.0

# For Colab, you might need to restart the runtime after installation.
# If running locally, ensure you're in your virtual environment.

print("Required libraries installed successfully!")


## High-Level System Overview: The E-commerce Assistant Architecture

Our agentic e-commerce assistant will be designed to handle three primary categories of customer interactions:

1.  **Product Q&A:** Answering specific questions about product features, specifications, compatibility, availability, and pricing. This requires deep knowledge of the product catalog.
2.  **Recommendations:** Providing personalized product suggestions based on user preferences, browsing history, purchase patterns, or the current context of their query. This moves beyond simple search to proactive assistance.
3.  **Order Queries:** Assisting with questions related to existing orders, such as status updates, shipping details, return policies, and cancellation procedures. This involves interacting with backend order management systems.

### The Multi-Agent Approach

To effectively manage these diverse tasks, we will employ a multi-agent architecture orchestrated by AutoGen. This allows us to decompose complex problems into smaller, manageable sub-tasks, each handled by a specialized agent. This modularity enhances robustness, scalability, and maintainability.

Imagine the following flow:

```mermaid
graph TD
    A[Customer Query] --> B(User Proxy Agent)
    B --> C{Orchestrator Agent}
    C -- "Product Info?" --> D[Product Info Agent]
    C -- "Recommendation?" --> E[Recommendation Agent]
    C -- "Order Status?" --> F[Order Management Agent]

    D -- "Query Product Catalog (RAG)" --> G[Vector DB / Product DB]
    E -- "Access User Profile / Product Embeddings" --> H[Recommendation Engine / DB]
    F -- "Call Order API" --> I[Order Management System]

    G --> D
    H --> E
    I --> F

    D --> J[Response to Customer]
    E --> J
    F --> J
    J --> A

    style A fill:#f9f,stroke:#333,stroke-width:2px
    style B fill:#bbf,stroke:#333,stroke-width:2px
    style C fill:#ccf,stroke:#333,stroke-width:2px
    style D fill:#dfd,stroke:#333,stroke-width:2px
    style E fill:#dfd,stroke:#333,stroke-width:2px
    style F fill:#dfd,stroke:#333,stroke-width:2px
    style G fill:#ffb,stroke:#333,stroke-width:2px
    style H fill:#ffb,stroke:#333,stroke-width:2px
    style I fill:#ffb,stroke:#333,stroke-width:2px
    style J fill:#f9f,stroke:#333,stroke-width:2px
```

*   **User Proxy Agent:** This agent acts as the customer's interface, receiving queries and relaying responses.
*   **Orchestrator Agent:** The brain of the operation. It analyzes the incoming query, determines its intent (Q&A, recommendation, or order), and routes it to the most appropriate specialized agent.
*   **Specialized Agents:**
    *   `ProductInfoAgent`: Expert in product details, leveraging RAG to retrieve information from the product catalog.
    *   `RecommendationAgent`: Skilled in suggesting relevant products, potentially using collaborative filtering or content-based methods.
    *   `OrderManagementAgent`: Connects to the e-commerce backend to fetch and process order-related information.
*   **Tools/Databases:** These are the external systems and data sources that agents interact with (e.g., vector databases for RAG, product databases, order APIs, recommendation engines).

This modular design ensures that each agent is highly focused on its domain, making the system robust and easier to develop and debug. AutoGen's powerful conversation framework will manage the communication and collaboration between these agents.


## Core Components in Detail

Let's break down the essential components that will form our agentic e-commerce assistant:

### 1. Large Language Models (LLMs)

At the heart of every agent is an LLM. These models provide the reasoning capabilities, natural language understanding, and generation power that allow agents to interpret queries, formulate plans, and generate coherent responses. We'll configure our agents to use powerful models like OpenAI's GPT series, or other compatible models via LiteLLM for flexibility.

### 2. AutoGen Framework

AutoGen is our chosen multi-agent conversation framework. It provides:

*   **Agent Abstraction:** Easy definition of different agent types (User Proxy, Assistant, etc.).
*   **Flexible Communication:** Agents can converse, delegate tasks, and provide feedback to each other.
*   **Tool Integration:** Seamlessly connect agents to external functions, APIs, and databases.
*   **Group Chat:** Facilitates complex workflows where multiple agents collaborate to solve a problem.

### 3. Retrieval Augmented Generation (RAG) System

For **Product Q&A**, a robust RAG system is critical. This involves:

*   **Product Catalog:** A comprehensive dataset of all products, including descriptions, specifications, images, and reviews.
*   **Embedding Model:** To convert product text into numerical vectors (embeddings).
*   **Vector Database (e.g., Qdrant):** To store and efficiently search these product embeddings, allowing for semantic retrieval of relevant product information based on customer queries.
*   **Retrieval Strategy:** Techniques to fetch the most relevant chunks of information from the vector database.

### 4. E-commerce Backend Integrations

For **Order Queries** and potentially **Recommendations**, agents will need to interact with real-world e-commerce systems:

*   **Order Management System (OMS) API:** To fetch order status, tracking information, and initiate actions like cancellations or returns.
*   **Product Database:** For structured product data that might not be suitable for RAG (e.g., exact stock levels, pricing).
*   **Recommendation Engine API:** If an existing recommendation system is in place, agents can query it for personalized suggestions.

### 5. User Profile & Interaction History

For truly personalized **Recommendations**, agents will need access to:

*   **User Profiles:** Storing preferences, past purchases, and demographic information.
*   **Interaction History:** Tracking browsing behavior, previously viewed products, and past queries to build a dynamic understanding of the user's intent and interests.

By combining these components, we'll build an intelligent, adaptable, and highly effective e-commerce assistant.


In [ ]:
import autogen
import os
from dotenv import load_dotenv

# Load environment variables from a .env file
# In Colab, you can upload a .env file or set secrets directly.
load_dotenv()

# --- Configuration for LLM --- 
# We'll use OpenAI's models for this project. Ensure OPENAI_API_KEY is set in your .env file.
# For 2026, we assume 'gpt-4o' (or its successor) is the leading general-purpose model.
config_list = [
    {
        "model": "gpt-4o", # Or "gpt-4-turbo", "gpt-3.5-turbo" for cost-effectiveness
        "api_key": os.getenv("OPENAI_API_KEY"),
    }
]

# --- Initialize a basic AutoGen environment --- 
# This is a 'hello world' to confirm AutoGen is working and can communicate with the LLM.

# 1. Create an AssistantAgent
# This agent is designed to be helpful and can execute code if given the capability.
assistant = autogen.AssistantAgent(
    name="assistant",
    llm_config={
        "config_list": config_list,
        "temperature": 0.7, # Controls randomness of output
    },
    system_message="You are a helpful AI assistant. Your goal is to provide concise and accurate answers."
)

# 2. Create a UserProxyAgent
# This agent represents the human user. It can send messages and execute code (if enabled).
user_proxy = autogen.UserProxyAgent(
    name="user_proxy",
    human_input_mode="NEVER", # Set to "ALWAYS" for interactive chat
    max_consecutive_auto_reply=10, # Max number of auto-replies before human intervention
    is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
    code_execution_config={
        "work_dir": "coding", # Directory for code execution
        "use_docker": False, # Set to True for isolated code execution (requires Docker)
    },
    llm_config={
        "config_list": config_list,
        "temperature": 0.7,
    }
)

# --- Start a simple conversation --- 
# This will test if the agents can communicate and if the LLM is accessible.
print("\n--- Starting a test conversation ---")
user_proxy.initiate_chat(
    assistant,
    message="Hello, assistant! What is the capital of France?",
)

print("\n--- Basic AutoGen setup complete and tested! ---")
print("If you saw a response to 'What is the capital of France?', your environment is correctly configured.")
